In [1]:
# =========================================
# 3.4 Integration — Create & Use semantic_codebook_full.json
# Steps:
#   1) Write a full semantic codebook JSON to prep_outputs/semantic_codebook_full.json
#   2) Load it and generate *_LABEL columns for coded variables
#   3) Save:
#        - prep_outputs/final_integrated_table_semantic.csv
#        - prep_outputs/semantic_dictionary.csv
#        - prep_outputs/semantic_codebook_snapshot.json
# =========================================

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------
# Paths
# -----------------------------
IN_CSV   = Path("prep_outputs/final_modeling_table.csv")
OUT_DIR  = Path("prep_outputs")
CODEBOOK_JSON = OUT_DIR / "semantic_codebook_full.json"
OUT_CSV  = OUT_DIR / "final_integrated_table_semantic.csv"
DICT_CSV = OUT_DIR / "semantic_dictionary.csv"
SNAPSHOT = OUT_DIR / "semantic_codebook_snapshot.json"

# Ensure paths
assert IN_CSV.exists(), f"Input CSV not found: {IN_CSV}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1) Create FULL semantic codebook JSON (edit/extend as needed)
#    Keys are strings to be valid JSON; values are labels.
#    Only coded variables are listed; continuous variables are intentionally omitted.
# -----------------------------
CODEBOOK = {
  # --- Demographics ---
  "SEX": { "1": "Male", "2": "Female" },
  "REGION": { "1": "Northeast", "2": "Midwest", "3": "South", "4": "West" },
  "R_MARITL": {
    "0": "Under 14 years",
    "1": "Married - spouse in household",
    "2": "Married - spouse not in household",
    "3": "Married - spouse in household unknown",
    "4": "Widowed",
    "5": "Divorced",
    "6": "Separated",
    "7": "Never married",
    "8": "Living with partner",
    "9": "Unknown"
  },
  "HISPAN_I": {
    "0":"Multiple Hispanic",
    "1":"Puerto Rico",
    "2":"Mexican",
    "3":"Mexican-American",
    "4":"Cuban",
    "5":"Dominican",
    "6":"Central/South American",
    "7":"Other Latin American, n.s.",
    "8":"Other Spanish",
    "9":"Hispanic/Spanish, n.s.",
    "10":"Type refused",
    "11":"Type not ascertained",
    "12":"Not Hispanic/Spanish"
  },
  "RACERPI2": {
    "1":"White only",
    "2":"Black/African American only",
    "3":"American Indian/Alaska Native only",
    "4":"Asian only",
    "5":"Race group not releasable",
    "6":"Multiple race"
  },

  # --- Employment / work status ---
  "DOINGLWA": {
    "1":"Working for pay",
    "2":"With a job, not at work",
    "3":"Looking for work",
    "4":"Unpaid in family business",
    "5":"Not working & not looking"
  },
  "WRKLYR4": {
    "0":"Had job last week",
    "1":"No job last week, had job in last 12m",
    "2":"No job last week, no job in last 12m",
    "3":"Never worked"
  },

  # --- Alcohol / smoking status ---
  "ALCSTAT": {
    "1":"Lifetime abstainer", "2":"Former infrequent", "3":"Former regular",
    "4":"Current infrequent", "5":"Current light", "6":"Current moderate",
    "7":"Refused", "8":"Not ascertained", "9":"Don't know", "10":"Unknown status"
  },
  "ALC12MTP": { "1":"Day(s)", "2":"Week(s)", "3":"Month(s)", "4":"Year(s)" },
  "SMKSTAT2": {
    "1":"Current every day smoker",
    "2":"Current some day smoker",
    "3":"Former smoker",
    "4":"Never smoker",
    "5":"Smoker, current status unknown",
    "9":"Unknown if ever smoked"
  },
  
  
   "ALCHRONR": { "0":"Not at all difficult", "1":"Only a little difficult", "2":"Somewhat difficult", "3":"Very difficult", "4":"Can't do at all","6":"Do not do this activity" },

  # --- Binary health history (1=Yes, 2=No) ---
  "HYPEV": { "1":"Yes", "2":"No" },
  "CHDEV": { "1":"Yes", "2":"No" },
  "MIEV":  { "1":"Yes", "2":"No" },
  "HRTEV": { "1":"Yes", "2":"No" },
  "STREV": { "1":"Yes", "2":"No" },
  "EPHEV": { "1":"Yes", "2":"No" },
  "COPDEV": { "1":"Yes", "2":"No" },
  "AASMEV": { "1":"Yes", "2":"No" },
  "CANEV": { "1":"Yes", "2":"No" },
  "AHAYFYR": { "1":"Yes", "2":"No" },
  "PAINLB": { "1":"Yes", "2":"No" },
  "PAINFACE": { "1":"Yes", "2":"No" },
  "SINYR": { "1":"Yes", "2":"No" },
  "AMIGR": { "1":"Yes", "2":"No" },
  "ANX_2": { "1":"Yes", "2":"No" },
  "DEP_2": { "1":"Yes", "2":"No" },

  # --- Ordinal / Likert (neutral labels; replace with official text if desired) ---
  "DEP_1":   { "1":"Daily", "2":"Weekly", "3":"Monthly", "4":"A few times a year", "5":"Never" },
  "ANX_1":   { "1":"Daily", "2":"Weekly", "3":"Monthly", "4":"A few times a year", "5":"Never" },
  "ASICCOLL":  { "1":"Very worried", "2":"Moderately worried", "3":"Not too worried","4":"Not worried at all","5":"Not Apply" },
  "TIRED_1": { "1":"Never", "2":"Some days", "3":"Most days", "4":"Every day" },
  "TIRED_2": { "1":"Some of the day", "2":"Most of the day", "3":"All of the day" },
  "TIRED_3": { "1":"A little", "2":"A lot", "3":"Somewhere in between a little and a lot" },
  "PAIN_2A": { "1":"Never", "2":"Some days", "3":"Most days", "4":"Every day" },
  "PAIN_4":  { "1":"A little", "2":"A lot", "3":"Somewhere in between a little and a lot" },
  "ANX_3R":  { "1":"A little", "2":"A lot", "3":"Somewhere in between a little and a lot" },

  "PAINECK": { "1":"Yes", "2":"No" },
  "JNTSYMP": { "1":"Yes", "2":"No" },
  "DIBPRE2": { "1":"Yes", "2":"No" },
  "SHTHEPB": { "1":"Yes", "2":"No" },
  "HIT1A":   { "1":"Yes", "2":"No" },
  "FLA1AR":  { "1":"Yes", "2":"No" },

  "AHCAFYR1": { "1":"Yes", "2":"No" },
  "ASIEFFRT": { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "ASIHOPLS": { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "AHCDLYR2": { "1":"Yes", "2":"No" },
  "AHCSYR1":  { "1":"Yes", "2":"No" },
  "AHCSYR6":  { "1":"Yes", "2":"No" },


  "ARX12_2":  { "1":"Yes", "2":"No" },
  "ARX12_3":  { "1":"Yes", "2":"No" },

  "ASIHIVT":  { "1":"Yes", "2":"No" },
 
  "ASIMEDC":  { "1":"Very worried", "2":"Moderately worried", "3":"Not too worried","4":"Not worried at all" },
  "ASINBILL": { "1":"Very worried", "2":"Moderately worried", "3":"Not too worried","4":"Not worried at all" },
  "ASINERV":  { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "ASIRETR": { "1":"Very worried", "2":"Moderately worried", "3":"Not too worried","4":"Not worried at all" },
  "ASIRSTLS": { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "ASISAD":  { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "ASISTLV":  { "1":"Very worried", "2":"Moderately worried", "3":"Not too worried","4":"Not worried at all" },
  "ASITENUR": { "1":"Less than 1 year", "2":"1-3 years", "3":"4-10 years", "4":"11-20 years", "5":"More than 20 years" },
  "ASIWTHLS": { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "AWORPAY": { "1":"Very worried", "2":"Somewhat worried","3":"Not at all worried" },
  "PAR_STAT": { "1":"Yes, the Sample Adult is a parent of a child residing in the family", "2":"There are minor children residing in the family but the Sample Adult is not their parent", "3":"There are no minor children residing in the family" }
}

# Write JSON file
with open(CODEBOOK_JSON, "w") as f:
  json.dump(CODEBOOK, f, indent=2)
print(f"[OK] Wrote semantic codebook -> {CODEBOOK_JSON}")

# -----------------------------
# 2) Load data and codebook, apply mapping to create *_LABEL columns
# -----------------------------
df = pd.read_csv(IN_CSV)
with open(CODEBOOK_JSON, "r") as f:
  codebook = json.load(f)

df_out = df.copy()
dict_rows = []

def map_one_column(frame: pd.DataFrame, col: str, mapping: dict):
    """Create a *_LABEL string column by mapping integer codes to labels."""
    # Convert JSON string keys to int where possible
    mapping_int = {}
    for k, v in mapping.items():
        try:
            mapping_int[int(k)] = v
        except Exception:
            mapping_int[k] = v
    s_num = pd.to_numeric(frame[col], errors="coerce")
    frame[f"{col}_LABEL"] = s_num.map(mapping_int).astype("string")
    # Collect dictionary rows
    for k, v in mapping_int.items():
        dict_rows.append({"column": col, "code": k, "label": v})

mapped_cols = []
for col, mapping in codebook.items():
    if col in df_out.columns:
        map_one_column(df_out, col, mapping)
        mapped_cols.append(col)

# -----------------------------
# 3) Save outputs
# -----------------------------
df_out.to_csv(OUT_CSV, index=False)
pd.DataFrame(dict_rows).to_csv(DICT_CSV, index=False)
with open(SNAPSHOT, "w") as f:
    json.dump(codebook, f, indent=2)

print(f"[OK] Semantic table -> {OUT_CSV} | shape: {df_out.shape}")
print(f"[OK] Dictionary     -> {DICT_CSV} | rows: {len(dict_rows)}")
print(f"[OK] Snapshot JSON  -> {SNAPSHOT}")
print(f"[Mapped] {len(mapped_cols)} columns mapped with labels.")

# -----------------------------
# 4) Optional: flag likely-coded columns not in the JSON (for you to extend)
# -----------------------------
low_card_candidates = []
for c in df.columns:
    if c in codebook:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        nunq = df[c].nunique(dropna=True)
        if 1 < nunq <= 12:
            low_card_candidates.append((c, nunq))

if low_card_candidates:
    print("\n[Review] Numeric low-cardinality columns NOT in codebook (unique count shown):")
    for name, nunq in sorted(low_card_candidates, key=lambda x: x[0]):
        print(f"  - {name}: {nunq}")
else:
    print("\n[Review] No obvious missing coded columns.")


[OK] Wrote semantic codebook -> prep_outputs/semantic_codebook_full.json
[OK] Semantic table -> prep_outputs/final_integrated_table_semantic.csv | shape: (25403, 103)
[OK] Dictionary     -> prep_outputs/semantic_dictionary.csv | rows: 159
[OK] Snapshot JSON  -> prep_outputs/semantic_codebook_snapshot.json
[Mapped] 44 columns mapped with labels.

[Review] Numeric low-cardinality columns NOT in codebook (unique count shown):
  - PAIN_INDEX: 7
  - SLEEP_SUFFICIENT: 2


In [ ]:
# =========================================
# 3.4 Integration — Create & Use semantic_codebook_full.json
# Steps:
#   1) Write a full semantic codebook JSON to prep_outputs/semantic_codebook_full.json
#   2) Load it and generate *_LABEL columns for coded variables
#   3) Save:
#        - prep_outputs/final_integrated_table_semantic.csv
#        - prep_outputs/semantic_dictionary.csv
#        - prep_outputs/semantic_codebook_snapshot.json
# =========================================

import os
import json
import numpy as np
import pandas as pd
from pathlib import Path

# -----------------------------
# Paths
# -----------------------------
IN_CSV   = Path("prep_outputs/final_modeling_table.csv")
OUT_DIR  = Path("prep_outputs")
CODEBOOK_JSON = OUT_DIR / "semantic_codebook_full.json"
OUT_CSV  = OUT_DIR / "final_integrated_table_semantic.csv"
DICT_CSV = OUT_DIR / "semantic_dictionary.csv"
SNAPSHOT = OUT_DIR / "semantic_codebook_snapshot.json"

# Ensure paths
assert IN_CSV.exists(), f"Input CSV not found: {IN_CSV}"
OUT_DIR.mkdir(parents=True, exist_ok=True)

# -----------------------------
# 1) Create FULL semantic codebook JSON (edit/extend as needed)
#    Keys are strings to be valid JSON; values are labels.
#    Only coded variables are listed; continuous variables are intentionally omitted.
# -----------------------------
CODEBOOK = {
  # --- Demographics ---
  "SEX": { "1": "Male", "2": "Female" },
  "REGION": { "1": "Northeast", "2": "Midwest", "3": "South", "4": "West" },
  "R_MARITL": {
    "0": "Under 14 years",
    "1": "Married - spouse in household",
    "2": "Married - spouse not in household",
    "3": "Married - spouse in household unknown",
    "4": "Widowed",
    "5": "Divorced",
    "6": "Separated",
    "7": "Never married",
    "8": "Living with partner",
    "9": "Unknown"
  },
  "HISPAN_I": {
    "0":"Multiple Hispanic",
    "1":"Puerto Rico",
    "2":"Mexican",
    "3":"Mexican-American",
    "4":"Cuban",
    "5":"Dominican",
    "6":"Central/South American",
    "7":"Other Latin American, n.s.",
    "8":"Other Spanish",
    "9":"Hispanic/Spanish, n.s.",
    "10":"Type refused",
    "11":"Type not ascertained",
    "12":"Not Hispanic/Spanish"
  },
  "RACERPI2": {
    "1":"White only",
    "2":"Black/African American only",
    "3":"American Indian/Alaska Native only",
    "4":"Asian only",
    "5":"Race group not releasable",
    "6":"Multiple race"
  },

  # --- Employment / work status ---
  "DOINGLWA": {
    "1":"Working for pay",
    "2":"With a job, not at work",
    "3":"Looking for work",
    "4":"Unpaid in family business",
    "5":"Not working & not looking"
  },
  "WRKLYR4": {
    "0":"Had job last week",
    "1":"No job last week, had job in last 12m",
    "2":"No job last week, no job in last 12m",
    "3":"Never worked"
  },

  # --- Alcohol / smoking status ---
  "ALCSTAT": {
    "1":"Lifetime abstainer", "2":"Former infrequent", "3":"Former regular",
    "4":"Current infrequent", "5":"Current light", "6":"Current moderate",
    "7":"Refused", "8":"Not ascertained", "9":"Don't know", "10":"Unknown status"
  },
  "ALC12MTP": { "1":"Day(s)", "2":"Week(s)", "3":"Month(s)", "4":"Year(s)" },
  "SMKSTAT2": {
    "1":"Current every day smoker",
    "2":"Current some day smoker",
    "3":"Former smoker",
    "4":"Never smoker",
    "5":"Smoker, current status unknown",
    "9":"Unknown if ever smoked"
  },
  
  
   "ALCHRONR": { "0":"Not at all difficult", "1":"Only a little difficult", "2":"Somewhat difficult", "3":"Very difficult", "4":"Can't do at all","6":"Do not do this activity" },

  # --- Binary health history (1=Yes, 2=No) ---
  "HYPEV": { "1":"Yes", "2":"No" },
  "CHDEV": { "1":"Yes", "2":"No" },
  "MIEV":  { "1":"Yes", "2":"No" },
  "HRTEV": { "1":"Yes", "2":"No" },
  "STREV": { "1":"Yes", "2":"No" },
  "EPHEV": { "1":"Yes", "2":"No" },
  "COPDEV": { "1":"Yes", "2":"No" },
  "AASMEV": { "1":"Yes", "2":"No" },
  "CANEV": { "1":"Yes", "2":"No" },
  "AHAYFYR": { "1":"Yes", "2":"No" },
  "PAINLB": { "1":"Yes", "2":"No" },
  "PAINFACE": { "1":"Yes", "2":"No" },
  "SINYR": { "1":"Yes", "2":"No" },
  "AMIGR": { "1":"Yes", "2":"No" },
  "ANX_2": { "1":"Yes", "2":"No" },
  "DEP_2": { "1":"Yes", "2":"No" },

  # --- Ordinal / Likert (neutral labels; replace with official text if desired) ---
  "DEP_1":   { "1":"Daily", "2":"Weekly", "3":"Monthly", "4":"A few times a year", "5":"Never" },
  "ANX_1":   { "1":"Daily", "2":"Weekly", "3":"Monthly", "4":"A few times a year", "5":"Never" },
  "ASICCOLL":  { "1":"Very worried", "2":"Moderately worried", "3":"Not too worried","4":"Not worried at all","5":"Not Apply" },
  "TIRED_1": { "1":"Never", "2":"Some days", "3":"Most days", "4":"Every day" },
  "TIRED_2": { "1":"Some of the day", "2":"Most of the day", "3":"All of the day" },
  "TIRED_3": { "1":"A little", "2":"A lot", "3":"Somewhere in between a little and a lot" },
  "PAIN_2A": { "1":"Never", "2":"Some days", "3":"Most days", "4":"Every day" },
  "PAIN_4":  { "1":"A little", "2":"A lot", "3":"Somewhere in between a little and a lot" },
  "ANX_3R":  { "1":"A little", "2":"A lot", "3":"Somewhere in between a little and a lot" },

  "PAINECK": { "1":"Yes", "2":"No" },
  "JNTSYMP": { "1":"Yes", "2":"No" },
  "DIBPRE2": { "1":"Yes", "2":"No" },
  "SHTHEPB": { "1":"Yes", "2":"No" },
  "HIT1A":   { "1":"Yes", "2":"No" },
  "FLA1AR":  { "1":"Yes", "2":"No" },

  "AHCAFYR1": { "1":"Yes", "2":"No" },
  "ASIEFFRT": { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "ASIHOPLS": { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "AHCDLYR2": { "1":"Yes", "2":"No" },
  "AHCSYR1":  { "1":"Yes", "2":"No" },
  "AHCSYR6":  { "1":"Yes", "2":"No" },


  "ARX12_2":  { "1":"Yes", "2":"No" },
  "ARX12_3":  { "1":"Yes", "2":"No" },

  "ASIHIVT":  { "1":"Yes", "2":"No" },
 
  "ASIMEDC":  { "1":"Very worried", "2":"Moderately worried", "3":"Not too worried","4":"Not worried at all" },
  "ASINBILL": { "1":"Very worried", "2":"Moderately worried", "3":"Not too worried","4":"Not worried at all" },
  "ASINERV":  { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "ASIRETR": { "1":"Very worried", "2":"Moderately worried", "3":"Not too worried","4":"Not worried at all" },
  "ASIRSTLS": { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "ASISAD":  { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "ASISTLV":  { "1":"Very worried", "2":"Moderately worried", "3":"Not too worried","4":"Not worried at all" },
  "ASITENUR": { "1":"Less than 1 year", "2":"1-3 years", "3":"4-10 years", "4":"11-20 years", "5":"More than 20 years" },
  "ASIWTHLS": { "1":"ALL of the time", "2":"MOST of the time", "3":"SOME of the time", "4":"A LITTLE of the time", "5":"NONE of the time"},
  "AWORPAY": { "1":"Very worried", "2":"Somewhat worried","3":"Not at all worried" },
  "PAR_STAT": { "1":"Yes, the Sample Adult is a parent of a child residing in the family", "2":"There are minor children residing in the family but the Sample Adult is not their parent", "3":"There are no minor children residing in the family" }
}

# Write JSON file
with open(CODEBOOK_JSON, "w") as f:
  json.dump(CODEBOOK, f, indent=2)
print(f"[OK] Wrote semantic codebook -> {CODEBOOK_JSON}")

# -----------------------------
# 2) Load data and codebook, apply mapping to create *_LABEL columns
# -----------------------------
df = pd.read_csv(IN_CSV)
with open(CODEBOOK_JSON, "r") as f:
  codebook = json.load(f)

df_out = df.copy()
dict_rows = []

def map_one_column(frame: pd.DataFrame, col: str, mapping: dict):
    """Create a *_LABEL string column by mapping integer codes to labels."""
    # Convert JSON string keys to int where possible
    mapping_int = {}
    for k, v in mapping.items():
        try:
            mapping_int[int(k)] = v
        except Exception:
            mapping_int[k] = v
    s_num = pd.to_numeric(frame[col], errors="coerce")
    frame[f"{col}_LABEL"] = s_num.map(mapping_int).astype("string")
    # Collect dictionary rows
    for k, v in mapping_int.items():
        dict_rows.append({"column": col, "code": k, "label": v})

mapped_cols = []
for col, mapping in codebook.items():
    if col in df_out.columns:
        map_one_column(df_out, col, mapping)
        mapped_cols.append(col)

# -----------------------------
# 3) Save outputs
# -----------------------------
df_out.to_csv(OUT_CSV, index=False)
pd.DataFrame(dict_rows).to_csv(DICT_CSV, index=False)
with open(SNAPSHOT, "w") as f:
    json.dump(codebook, f, indent=2)

print(f"[OK] Semantic table -> {OUT_CSV} | shape: {df_out.shape}")
print(f"[OK] Dictionary     -> {DICT_CSV} | rows: {len(dict_rows)}")
print(f"[OK] Snapshot JSON  -> {SNAPSHOT}")
print(f"[Mapped] {len(mapped_cols)} columns mapped with labels.")

# -----------------------------
# 4) Optional: flag likely-coded columns not in the JSON (for you to extend)
# -----------------------------
low_card_candidates = []
for c in df.columns:
    if c in codebook:
        continue
    if pd.api.types.is_numeric_dtype(df[c]):
        nunq = df[c].nunique(dropna=True)
        if 1 < nunq <= 12:
            low_card_candidates.append((c, nunq))

if low_card_candidates:
    print("\n[Review] Numeric low-cardinality columns NOT in codebook (unique count shown):")
    for name, nunq in sorted(low_card_candidates, key=lambda x: x[0]):
        print(f"  - {name}: {nunq}")
else:
    print("\n[Review] No obvious missing coded columns.")


[OK] Wrote semantic codebook -> prep_outputs/semantic_codebook_full.json
[OK] Semantic table -> prep_outputs/final_integrated_table_semantic.csv | shape: (25403, 103)
[OK] Dictionary     -> prep_outputs/semantic_dictionary.csv | rows: 159
[OK] Snapshot JSON  -> prep_outputs/semantic_codebook_snapshot.json
[Mapped] 44 columns mapped with labels.

[Review] Numeric low-cardinality columns NOT in codebook (unique count shown):
  - PAIN_INDEX: 7
  - SLEEP_SUFFICIENT: 2
